# 🧠 03 - Huấn luyện LayoutLMv3

Notebook này cấu hình HF Trainer để huấn luyện mô hình LayoutLMv3 cho bài toán nhận dạng thực thể hóa đơn (KIE).

**Nội dung:**
1. Setup môi trường
2. Load Dataset
3. Khởi tạo mô hình
4. Huấn luyện (HF Trainer)
5. Đánh giá mô hình
6. Inference demo

## 1. Setup môi trường

In [ ]:
# Chạy file setup_colab.sh để tự động cài đặt mọi thứ
!bash /content/document-ai-vn/setup_colab.sh

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
os.chdir('/content/document-ai-vn')
sys.path.insert(0, '/content/document-ai-vn')

## 2. Load Dataset

In [ ]:
from configs.paths import PROCESSED_DIR
from src.kie.dataset import build_hf_dataset, encode_example

print('Loading and encoding datasets...')

# Dataset thô
raw_train_ds = build_hf_dataset(str(PROCESSED_DIR / 'train.jsonl'))
raw_val_ds = build_hf_dataset(str(PROCESSED_DIR / 'val.jsonl'))

# Mã hóa qua LayoutLMv3Processor
train_ds = raw_train_ds.map(encode_example)
val_ds = raw_val_ds.map(encode_example)

print(f'Train features: {train_ds.features.keys()}')
print(f'Train size: {len(train_ds)}, Val size: {len(val_ds)}')

## 3. Khởi tạo mô hình

In [ ]:
from transformers import LayoutLMv3ForTokenClassification
from configs.layoutlmv3_config import MODEL_NAME, LABEL2ID, ID2LABEL

model = LayoutLMv3ForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

print(f'Loaded model {MODEL_NAME} với {len(LABEL2ID)} nhãn')

## 4. Huấn luyện (HF Trainer)

In [ ]:
from transformers import Trainer, TrainingArguments
from configs.paths import CHECKPOINT_DIR
from configs.layoutlmv3_config import (
    TRAIN_BATCH_SIZE,
    EVAL_BATCH_SIZE,
    LEARNING_RATE,
    NUM_EPOCHS,
    WEIGHT_DECAY,
)
from src.kie.evaluate import compute_metrics

args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    report_to='none', # Tắt wandb nếu không dùng
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print('Bắt đầu huấn luyện...')
trainer.train()

In [ ]:
# Lưu mô hình tốt nhất
import shutil

best_model_dir = CHECKPOINT_DIR / 'best_model'
if best_model_dir.exists():
    shutil.rmtree(best_model_dir)

trainer.save_model(str(best_model_dir))
print(f'Đã lưu best model tại {best_model_dir}')

## 5. Đánh giá mô hình

In [ ]:
eval_results = trainer.evaluate()
print('=== Kết quả đánh giá trên tập Val ===')
for k, v in eval_results.items():
    print(f'{k}: {v:.4f}')

## 6. Inference test trên ảnh thực tế

In [ ]:
from src.kie.infer import predict_from_image
import matplotlib.pyplot as plt
from PIL import Image

# Chọn một ảnh ngẫu nhiên từ tập xác thực (hoặc tự upload)
idx = 0
sample = raw_val_ds[idx]
image_path = sample['image_path']

print(f'Thực hiện inference cho ảnh: {image_path}')
prediction = predict_from_image(image_path)

print('\n=== Kết quả trích xuất ===')
for entity_type, extracted_text in prediction.items():
    print(f'- {entity_type}: {extracted_text}')
    
# Hiển thị ảnh kèm kết quả model
image = Image.open(image_path)
plt.figure(figsize=(10, 8))
plt.imshow(image)
plt.axis('off')
for i, (k, v) in enumerate(prediction.items()):
    plt.text(10, 20 + i*30, f'{k}: {v}', 
             backgroundcolor='white', color='red', fontsize=12,
             bbox=dict(facecolor='white', alpha=0.8))
plt.title('Kết quả Inference')
plt.show()